# 招标文件分析 — 主流程完整测试

流程：**文档预处理 → RAG检索 → Policy提取 → Critic视觉核验 → 重写（如需）**

> 确保 notebook 在 `tender-deep-research/` 目录下运行

## 0. 变量配置
所有可调参数集中在此处，修改这里即可

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

# ── 文件路径 ──────────────────────────────────────────────────
PDF_PATH = "cache/uploads/政府采购货物类公开招标文件示范文本（试行）.pdf"

# ── 测试提取的标书关键词（2个）────────────────────────────────
# 注：该文件为示范空白模板，选取文件中实际存在的条款内容作为测试
KEYS = [
    "投标保证金比例",
    "合同当事人定义",
]

# ── 模型配置 ──────────────────────────────────────────────────
POLICY_MODEL   = "qwen-plus"          # 文本提取/重写 LLM
CRITIC_MODEL   = "qwen-vl-max"        # 视觉核验 VLM
EMBED_MODEL    = "text-embedding-v4"  # Embedding 模型
EMBED_DIM      = 1024                 # text-embedding-v4 默认维度

DASHSCOPE_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"

# ── RAG 参数 ──────────────────────────────────────────────────
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 64
TOP_K         = 5

# ── 流程参数 ──────────────────────────────────────────────────
MAX_ITERATIONS = 3

print("✅ 配置加载完成")
print(f"   PDF      : {PDF_PATH}")
print(f"   提取要素 : {KEYS}")
print(f"   Policy   : {POLICY_MODEL}")
print(f"   Critic   : {CRITIC_MODEL}")
print(f"   Embedding: {EMBED_MODEL}")

✅ 配置加载完成
   PDF      : cache/uploads/政府采购货物类公开招标文件示范文本（试行）.pdf
   提取要素 : ['投标保证金比例', '合同当事人定义']
   Policy   : qwen-plus
   Critic   : qwen-vl-max
   Embedding: text-embedding-v4


## 1. 初始化客户端
从 `.env` 读取 API Key，构建 LLM / VLM / Embedding 客户端

In [2]:
from utils.config_loader import load_config
from core.llm_client import LLMClient, VLMClient, EmbeddingClient

# 读取 .env 中的 API Key
CFG = load_config()
API_KEY = CFG["policy_llm"]["api_key"]

# 使用 notebook 顶部定义的模型变量，覆盖 config.yaml 中的设置
policy_llm = LLMClient({
    "api_base"   : DASHSCOPE_BASE,
    "api_key"    : API_KEY,
    "model"      : POLICY_MODEL,
    "temperature": 0.3,
    "max_tokens" : 4096,
})

critic_vlm = VLMClient({
    "api_base"   : DASHSCOPE_BASE,
    "api_key"    : API_KEY,
    "model"      : CRITIC_MODEL,
    "temperature": 0.1,
    "max_tokens" : 2048,
})

embed_client = EmbeddingClient({
    "api_base"  : DASHSCOPE_BASE,
    "api_key"   : API_KEY,
    "model"     : EMBED_MODEL,
    "dimensions": EMBED_DIM,
})

print("✅ 客户端初始化完成")
print(f"   Policy   : {policy_llm.api_base} / {policy_llm.model}")
print(f"   Critic   : {critic_vlm.api_base} / {critic_vlm.model}")
print(f"   Embedding: {embed_client.api_base} / {embed_client.model}")

✅ 客户端初始化完成
   Policy   : https://dashscope.aliyuncs.com/compatible-mode/v1 / qwen-plus
   Critic   : https://dashscope.aliyuncs.com/compatible-mode/v1 / qwen-vl-max
   Embedding: https://dashscope.aliyuncs.com/compatible-mode/v1 / text-embedding-v4


## 2. 文档预处理
使用 PyMuPDF 提取每页文本，并以 300 DPI 渲染为 PNG 图片（base64），供后续 VLM 使用

In [3]:
from core.doc_processor import process_pdf, chunk_text_by_pages

print(f"📄 正在解析 PDF: {PDF_PATH}")
pages_data = process_pdf(PDF_PATH, pages_dir="cache/pages")

total_pages = pages_data["total_pages"]
pages       = pages_data["pages"]

print(f"\n✅ 文档解析完成")
print(f"   总页数   : {total_pages}")
print(f"   首页文本 (前150字):")
print(f"   {pages[0]['text'][:150].strip()}...")

📄 正在解析 PDF: cache/uploads/政府采购货物类公开招标文件示范文本（试行）.pdf
[2026-03-18 18:18:11] INFO core.doc_processor: PDF 处理完成: 政府采购货物类公开招标文件示范文本（试行）.pdf，共 91 页

✅ 文档解析完成
   总页数   : 91
   首页文本 (前150字):
   政府采购货物类公开招标
文件示范文本（试行）
（征求意见稿）
采 购 人：
采购代理机构：
年
月...


## 3. 构建 RAG 向量索引
将文本切块 → 调用 `text-embedding-v4` 批量向量化 → 写入 FAISS 内存索引

In [4]:
from core.rag import RAGEngine

# 文本切块
chunks = chunk_text_by_pages(pages, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
print(f"📦 文本切块完成，共 {len(chunks)} 个块")
print(f"   示例块 (chunk_id=0): {chunks[0]['text'][:80].strip()}...")

# 构建索引
print(f"\n🧮 正在构建向量索引 (模型: {EMBED_MODEL})...")
rag = RAGEngine(embed_client, top_k=TOP_K)
await rag.build_index(chunks)

print(f"✅ 索引构建完成，共 {rag.index.ntotal} 个向量，维度 {EMBED_DIM}")

[2026-03-18 18:18:11] INFO core.doc_processor: 文本分块完成：172 个块
📦 文本切块完成，共 172 个块
   示例块 (chunk_id=0): 政府采购货物类公开招标
文件示范文本（试行）
（征求意见稿）
采 购 人：
采购代理机构：
年
月...

🧮 正在构建向量索引 (模型: text-embedding-v4)...
[2026-03-18 18:18:11] INFO core.rag: 构建向量索引，共 172 个文本块（批大小 10）…
[2026-03-18 18:18:34] INFO core.rag: 索引构建完成
✅ 索引构建完成，共 172 个向量，维度 1024


## 4. RAG 语义检索
对每个要素分别检索最相关的 top-k 文本块

In [5]:
print(f"🔎 正在检索 {len(KEYS)} 个要素...\n")
rag_results = await rag.search_for_keys(KEYS)

for key, hits in rag_results.items():
    pages_found = sorted({h['page_num'] for h in hits})
    print(f"  「{key}」→ 命中来源页: {pages_found}")
    print(f"   最高分块 (score={hits[0]['score']:.3f}):")
    print(f"   {hits[0]['text'][:100].strip()}...")
    print()

🔎 正在检索 2 个要素...

[2026-03-18 18:18:35] INFO core.rag: 「投标保证金比例」→ 在第 [9, 11, 17, 18, 24] 页找到相关内容
[2026-03-18 18:18:36] INFO core.rag: 「合同当事人定义」→ 在第 [50, 53, 55, 56, 61] 页找到相关内容
  「投标保证金比例」→ 命中来源页: [9, 11, 17, 18, 24]
   最高分块 (score=0.700):
   33.4 因重大变故，采购任务取消的；
33.5 法律、法规以及招标文件规定的其他废标情形。
十、签订合同和验收
34. 履约保证金
“投标人须知前附表”要求中标供应商提交履约保证金的，签订合同前，投...

  「合同当事人定义」→ 命中来源页: [50, 53, 55, 56, 61]
   最高分块 (score=0.606):
   - 50 -
第二节 政府采购合同通用条款
1. 定义
1.1合同当事人
（1）采购人（以下称甲方）是指使用财政性资金，通过政府采购方式向供应商购买货
物及其相关服务的国家机关、事业单位、团体组织。...



## 5. 迭代流程：Policy 提取 → Critic 核验 → Rewrite
最多执行 `MAX_ITERATIONS` 轮，Critic 全部通过后提前收敛

In [6]:
from core.policy import PolicyAgent
from core.critic import CriticAgent

policy_agent = PolicyAgent(policy_llm)
critic_agent = CriticAgent(critic_vlm)

items     = []
feedbacks = []
converged = False

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f"{'='*50}")
    print(f"🔄 第 {iteration} 轮")
    print(f"{'='*50}")

    # ── Policy 提取 / 重写 ────────────────────────────────────
    if iteration == 1:
        print("\n📝 [Policy] 首次提取要素...")
        items = await policy_agent.extract(KEYS, rag_results)
    else:
        print("\n📝 [Policy] 根据 Critic 反馈重写...")
        items = await policy_agent.rewrite(KEYS, rag_results, items, feedbacks)

    print("   提取结果：")
    for item in items:
        print(f"   · 「{item.key}」= {item.value}  "
              f"(第{item.source_page}页, 置信度:{item.confidence:.0%})")

    # ── Critic 视觉核验 ───────────────────────────────────────
    print("\n👁️  [Critic] 视觉核验...")
    feedbacks = await critic_agent.verify(items, pages)

    print("   核验结果：")
    for fb in feedbacks:
        if fb.verified:
            print(f"   ✅ 「{fb.key}」确认正确  — {fb.comment}")
        else:
            print(f"   ❌ 「{fb.key}」有误，实际值: {fb.actual_value}  — {fb.comment}")

    # ── 收敛判断 ──────────────────────────────────────────────
    failed = [fb for fb in feedbacks if not fb.verified]
    if not failed:
        converged = True
        print(f"\n🎉 第 {iteration} 轮全部通过，提前收敛！")
        break
    else:
        print(f"\n⚠️  {len(failed)} 个要素未通过，进入下一轮...")

if not converged:
    print(f"\n⚠️  已达最大迭代次数 {MAX_ITERATIONS} 轮")

🔄 第 1 轮

📝 [Policy] 首次提取要素...
[2026-03-18 18:18:42] INFO core.policy: 提取「投标保证金比例」= 2% (第18页, conf=0.95)
[2026-03-18 18:18:42] INFO core.policy: 提取「合同当事人定义」= （1）采购人（以下称甲方）是指使用财政性资金，通过政府采购方式向供应商购买货物及其相关服务的国家机关、事业单位、团体组织。
（2）供应商（以下称乙方）是指参加政府采购活动并且中标（成交），向采购人提供合同约定的货物及其相关服务的法人、非法人组织或者自然人。 (第55页, conf=0.98)
   提取结果：
   · 「投标保证金比例」= 2%  (第18页, 置信度:95%)
   · 「合同当事人定义」= （1）采购人（以下称甲方）是指使用财政性资金，通过政府采购方式向供应商购买货物及其相关服务的国家机关、事业单位、团体组织。
（2）供应商（以下称乙方）是指参加政府采购活动并且中标（成交），向采购人提供合同约定的货物及其相关服务的法人、非法人组织或者自然人。  (第55页, 置信度:98%)

👁️  [Critic] 视觉核验...
[2026-03-18 18:18:44] INFO core.critic: ✅ 「投标保证金比例」
[2026-03-18 18:18:47] INFO core.critic: ✅ 「合同当事人定义」
   核验结果：
   ✅ 「投标保证金比例」确认正确  — 提取值2%与原文一致，正确
   ✅ 「合同当事人定义」确认正确  — 信息与图片内容一致

🎉 第 1 轮全部通过，提前收敛！


## 6. 最终结果汇总

In [7]:
# 更新 verified 状态
verified_keys = {fb.key for fb in feedbacks if fb.verified}
for item in items:
    if item.key in verified_keys:
        item.verified = True

print("## 📋 最终提取结果\n")
print(f"{'要素':<14} {'提取值':<20} {'来源页':<8} {'核验'}")
print("-" * 55)
for item in items:
    mark = "✅" if item.verified else "⚠️ "
    page = f"第{item.source_page}页" if item.source_page else "—"
    val  = (item.value or "未找到")[:18]
    print(f"{item.key:<14} {val:<20} {page:<8} {mark}")

print()
print(f"共迭代 {iteration} 轮，{'已完全收敛 ✅' if converged else '达到最大迭代次数 ⚠️'}")

## 📋 最终提取结果

要素             提取值                  来源页      核验
-------------------------------------------------------
投标保证金比例        2%                   第18页     ✅
合同当事人定义        （1）采购人（以下称甲方）是指使用财   第55页     ✅

共迭代 1 轮，已完全收敛 ✅
